# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 14

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В чем практическая разница между обращением к данным через `.loc[]` и `.iloc[]` в библиотеке Pandas?
```python
val_a = df.loc[0, 'nickname']
val_b = df.iloc[0, 1]
```
По какому принципу находит элемент каждый из этих методов?

**Ответ:**
* `.loc[]` осуществляет доступ по **пользовательским именам и меткам** строк и колонок (label-based: название столбца 'nickname', исходная метка строки 0).
* `.iloc[]` осуществляет доступ строго по **целочисленным порядковым номерам позиций** (integer position-based: 0-я строка, 1-й по счету столбец).

---

#### 1.2. Аналитик выполнил две команды:
```python
cnt = df['city'].nunique()
lst = df['city'].unique()
```
Чем принципиально различаются результаты в переменных `cnt` и `lst`? Что возвращает число, а что — массив самих названий?

**Ответ:**
* `cnt = df['city'].nunique()` возвращает одно целое **число** — количество различных уникальных городов.
* `lst = df['city'].unique()` возвращает **массив/список** с самими уникальными названиями этих городов.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["net_revenue"] = (df["total_paid"] * 0.87).round(2)
```

**Ответ:**
Рассчитывает чистый доход клуба за вычетом налога/комиссии в размере 13%.

---

#### 1.4. Зачем после группировки данных в Pandas аналитики вызывают метод `.reset_index()`?
```python
summary = df.groupby('category')['total_amount'].sum().reset_index()
```
Как изменится структура таблицы `summary` при добавлении `.reset_index()`?

**Ответ:**
Метод `.reset_index()` переносит столбец группировки (`category`) из индекса обратно в обычные столбцы данных и возвращает таблице стандартный целочисленный индекс строк от 0. Без него `category` осталась бы индексом.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `bar_orders.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path("../variant_14")
df_audit = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Суммарное число проданных товаров: 1.61")

Количество строк в таблице: 350
Столбцы датафрейма: ['order_id', 'member_id', 'item_name', 'category', 'order_date', 'quantity', 'item_price', 'total_amount']
Суммарное число проданных товаров: 1.61

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_14")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,GTA V,30473,2.37
1,Atomic Heart,27160,2.31
2,Valorant,24720,2.10
3,Rust,24528,2.28
4,Apex Legends,24236,2.05
5,Warzone,22949,2.27
6,Counter-Strike 2,21505,2.11
7,PUBG,20327,1.96
8,Dota 2,20228,2.33
9,Cyberpunk 2077,19587,2.42


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` колонка `member_id` является внешним ключом (`FOREIGN KEY`), ссылающимся на таблицу `members`.
Зачем нужна эта связь и что произойдет, если попытаться добавить игровую сессию для несуществующего клиента с `member_id = 99999`?

**Ответ:**
Внешний ключ связывает дочернюю таблицу (`game_sessions`) с родительской (`members`) и обеспечивает **ссылочную целостность**. СУБД отклонит вставку сессии с несуществующим клиентом (`FOREIGN KEY constraint failed`), предотвращая появление «потерянных» записей.

---

#### 3.2. Чем случайное выполнение команды `DROP TABLE members;` опаснее для работы клуба, чем команда `DELETE FROM members;`?
Сможет ли программа регистрации сразу после этого вставить нового клиента?

**Ответ:**
`DELETE` удаляет только строки, сохраняя структуру таблицы, ее типы и ограничения (программа сможет сразу вставлять новых клиентов). `DROP TABLE` физически удаляет саму таблицу из схемы базы данных — приложение упадет с ошибкой «таблица не существует» (`no such table: members`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT city, COUNT(*) AS members_cnt, AVG(balance) AS avg_bal FROM members GROUP BY city ORDER BY members_cnt DESC;
```

**Ответ:**
Запрос группирует клиентов по городам (`city`) и вычисляет число клиентов (`COUNT(*)`) и их средний баланс (`AVG(balance)`), сортируя по убыванию числа клиентов.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_14")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT session_id, member_id, game_title, duration_hours, total_paid
FROM game_sessions
WHERE duration_hours >= 4.0 AND total_paid >= 800
ORDER BY total_paid DESC, session_id ASC
LIMIT 6;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,session_id,member_id,game_title,duration_hours,total_paid
0,20337,65,Rust,6.0,2400.0
1,20398,108,Rust,6.0,2400.0
2,20054,101,Cyberpunk 2077,5.0,2000.0
3,20255,57,Dota 2,5.0,1900.0
4,20014,87,Apex Legends,4.0,1600.0
5,20084,106,GTA V,4.0,1600.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT zone_name, COUNT(*) AS pcs_count, ROUND(AVG(hourly_rate), 2) AS avg_rate
FROM hardware_pcs
GROUP BY zone_name
HAVING COUNT(*) >= 5
ORDER BY avg_rate DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,zone_name,pcs_count,avg_rate
0,VIP,5,392.00
1,Bootcamp,8,252.50
2,Standard,14,144.29


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT p.pc_id, p.pc_number, COUNT(s.session_id) AS sessions_count, ROUND(SUM(s.total_paid), 2) AS total_paid
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE p.zone_name = 'Standard'
GROUP BY p.pc_id, p.pc_number
ORDER BY total_paid DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,pc_id,pc_number,sessions_count,total_paid
0,12,PC-12,26,8619.0
1,11,PC-11,21,8430.0
2,1,PC-01,23,7418.0
3,2,PC-02,22,7209.0
4,3,PC-03,26,6410.0
5,6,PC-06,19,6247.0
6,7,PC-07,21,6172.0
7,10,PC-10,18,5865.0
8,13,PC-13,18,5603.0
9,9,PC-09,16,5002.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT tournament_id, tournament_name, game_title, prize_fund, entry_fee
FROM tournaments
WHERE entry_fee > (SELECT AVG(entry_fee) FROM tournaments)
ORDER BY entry_fee DESC;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,tournament_id,tournament_name,game_title,prize_fund,entry_fee
0,106,Pro Masters Elite: CS2,Counter-Strike 2,120000.0,1000.0
1,102,Spring DOTA 2 Major,Dota 2,75000.0,700.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Сессии с учетом скидки

**Найденные ошибки:**
1. Использование псевдонима в `WHERE`: псевдоним `discounted_paid` еще не вычислен на этапе фильтрации строк, необходимо написать `total_paid * 0.9 > 400`.

In [9]:
query_51_fixed = '''
SELECT 
    session_id,
    member_id,
    game_title,
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE total_paid * 0.9 > 400
ORDER BY discounted_paid DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,session_id,member_id,game_title,discounted_paid
0,20337,65,Rust,2160.0
1,20398,108,Rust,2160.0
2,20054,101,Cyberpunk 2077,1800.0
3,20255,57,Dota 2,1710.0
4,20014,87,Apex Legends,1440.0
5,20084,106,GTA V,1440.0
6,20386,2,Rust,1440.0
7,20414,60,Dota 2,1368.0
8,20233,107,Cyberpunk 2077,1350.0
9,20489,62,Valorant,1350.0


#### 5.2. Анализ крупных расходов клиентов в баре

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает идентификатор клиента с номером заказа вместо `b.member_id`.
2. Синтаксис и alias: предложение `WHERE` размещено после `GROUP BY` и использует псевдоним `total_bar`, необходимо написать `HAVING SUM(b.total_amount) > 1500`.

In [10]:
query_52_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    SUM(b.total_amount) AS total_bar
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING SUM(b.total_amount) > 1500
ORDER BY total_bar DESC
LIMIT 10;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,member_id,nickname,city,total_bar
0,90,Wolf_983,Москва,12140.0
1,102,Shade_57,Москва,8160.0
2,103,Turbo_411,Москва,6570.0
3,101,Phantom_37,Москва,6000.0
4,99,Specter_892,Москва,5870.0
5,74,Reaper_670,Казань,5690.0
6,92,Grizzly_116,Москва,5520.0
7,134,Viper_185,Екатеринбург,5345.0
8,112,Slayer_878,Санкт-Петербург,5155.0
9,51,Valkyrie_817,Москва,4845.0


#### 5.3. Анализ активности и расходов клиентов

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = s.pc_id` связывает идентификатор клиента с номером компьютера вместо `s.member_id`.
2. Агрегатная функция в `WHERE`: условие `WHERE SUM(s.total_paid) > 1200` должно быть перенесено в блок `HAVING`.
3. Неполный `GROUP BY`: столбцы `m.nickname` и `m.city` присутствуют в `SELECT`, но пропущены в списке группировки.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    COUNT(s.session_id) AS cnt,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING SUM(s.total_paid) > 1200
ORDER BY total_spent DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,cnt,total_spent
0,108,Lynx_33,Москва,6,4950.0
1,65,Phantom_416,Казань,7,4939.0
2,2,Storm_906,Казань,6,4895.0
3,36,Sniper_381,Казань,7,4815.0
4,112,Slayer_878,Санкт-Петербург,11,4690.0
5,80,Bullet_775,Новосибирск,8,4645.0
6,87,Paladin_693,Санкт-Петербург,6,4250.0
7,60,Cobra_22,Екатеринбург,6,4005.0
8,62,Slayer_889,Москва,7,3790.0
9,116,Kraken_18,Казань,5,3785.0
